# AI Skill Gap & Role Matching Engine


In [1]:
import pandas as pd
import os

BASE_DIR = os.path.abspath("..")

skills_path = os.path.join(BASE_DIR, "data", "processed", "skills_processed.csv")
role_skill_path = os.path.join(BASE_DIR, "data", "processed", "role_skill_map.csv")
demand_path = os.path.join(BASE_DIR, "data", "processed", "skill_demand_scores.csv")

skills_df = pd.read_csv(skills_path)
role_skill_map = pd.read_csv(role_skill_path)
skill_demand = pd.read_csv(demand_path)

skills_df.head()


,Title,ExperienceLevel,YearsOfExperience,Skills,Keywords,Processed_Skills,Normalized_Skill
0,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,c#,csharp
1,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,vb.net basics,vb.net basics
2,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,.net framework,.net framework
3,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,.net core fundamentals,.net core fundamentals
4,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,asp.net,aspnet


In [2]:
role_skill_map["skill_text"] = role_skill_map["Role_Skills"].apply(
    lambda x: " ".join(eval(x))
)

role_skill_map[["Title", "skill_text"]].head()


,Title,skill_text
0,.NET Developer,.net core basics git design patterns html kube...
1,AI Engineer - Experienced,kubernetes spark deep learning r c++ pytorch k...
2,AI Engineer - Fresher,c++ r pytorch keras nltk java clustering pytho...
3,AI Prompt Engineer,intro nlp openai gpt models exposure teamwork ...
4,AR/VR Developer,unreal engine basics teamwork vuforia communic...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
role_vectors = vectorizer.fit_transform(role_skill_map["skill_text"])


In [4]:
user_skills = ["python", "sql", "pandas"]
user_skill_text = " ".join(user_skills)

user_vector = vectorizer.transform([user_skill_text])


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(user_vector, role_vectors)[0]

role_skill_map["Match_Score"] = similarity_scores
role_skill_map.sort_values(by="Match_Score", ascending=False).head()


,Title,Role_Skills,skill_text,Match_Score
54,Data Analyst / Data Scientist Intern,"['matplotlib', 'python', 'r', 'seaborn', 'sql'...",matplotlib python r seaborn sql basic ml: clas...,0.427800
76,Fresher Data Scientist,"['matplotlib', 'python', 'r', 'seaborn', 'sql'...",matplotlib python r seaborn sql basic ml: clas...,0.412353
61,Data Scientist - Fresher,"['postgresql', 'matplotlib', 'python', 'r', 'b...",postgresql matplotlib python r basic ml algori...,0.382197
56,Data Science Associate,"['postgresql', 'matplotlib', 'python', 'r', 's...",postgresql matplotlib python r seaborn sql bas...,0.335231
59,Data Scientist - Entry Level,"['data wrangling', 'postgresql', 'matplotlib',...",data wrangling postgresql matplotlib python r ...,0.328205


In [6]:
role_skill_map["Match_Percentage"] = (role_skill_map["Match_Score"] * 100).round(2)
role_skill_map.head()


,Title,Role_Skills,skill_text,Match_Score,Match_Percentage
0,.NET Developer,"['.net core basics', 'git', 'design patterns',...",.net core basics git design patterns html kube...,0.020660,2.07
1,AI Engineer - Experienced,"['kubernetes', 'spark', 'deep learning', 'r', ...",kubernetes spark deep learning r c++ pytorch k...,0.024349,2.43
2,AI Engineer - Fresher,"['c++', 'r', 'pytorch', 'keras', 'nltk', 'java...",c++ r pytorch keras nltk java clustering pytho...,0.262617,26.26
3,AI Prompt Engineer,"['intro nlp', 'openai gpt models exposure', 't...",intro nlp openai gpt models exposure teamwork ...,0.024013,2.40
4,AR/VR Developer,"['unreal engine basics', 'teamwork', 'vuforia'...",unreal engine basics teamwork vuforia communic...,0.006366,0.64


In [7]:
def skill_gap(user_skills, role_skills):
    role_skills = set(role_skills)
    user_skills = set(user_skills)
    return list(role_skills - user_skills)


In [8]:
top_roles = role_skill_map.sort_values(
    by="Match_Percentage", ascending=False
).head(3)

top_roles["Missing_Skills"] = top_roles["Role_Skills"].apply(
    lambda x: skill_gap(user_skills, eval(x))
)

top_roles[["Title", "Match_Percentage", "Missing_Skills"]]


,Title,Match_Percentage,Missing_Skills
54,Data Analyst / Data Scientist Intern,42.78,"[seaborn, basic ml: classification, clustering..."
76,Fresher Data Scientist,41.24,"[basic ml: classification, regression, cluster..."
61,Data Scientist - Fresher,38.22,"[seaborn, mysql, basic ml algorithms: regressi..."


In [9]:
demand_dict = dict(
    zip(skill_demand["Skill"], skill_demand["Demand_Score"])
)

def weighted_gap_score(missing_skills):
    return sum(demand_dict.get(skill, 0) for skill in missing_skills)

top_roles["Gap_Severity_Score"] = top_roles["Missing_Skills"].apply(weighted_gap_score)
top_roles


,Title,Role_Skills,skill_text,Match_Score,Match_Percentage,Missing_Skills,Gap_Severity_Score
54,Data Analyst / Data Scientist Intern,"['matplotlib', 'python', 'r', 'seaborn', 'sql'...",matplotlib python r seaborn sql basic ml: clas...,0.427800,42.78,"[seaborn, basic ml: classification, clustering...",1.118182
76,Fresher Data Scientist,"['matplotlib', 'python', 'r', 'seaborn', 'sql'...",matplotlib python r seaborn sql basic ml: clas...,0.412353,41.24,"[basic ml: classification, regression, cluster...",1.127273
61,Data Scientist - Fresher,"['postgresql', 'matplotlib', 'python', 'r', 'b...",postgresql matplotlib python r basic ml algori...,0.382197,38.22,"[seaborn, mysql, basic ml algorithms: regressi...",1.109091
